# Run inference for EarTTS

In [ ]:
from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM
from vllm.model_executor.models.eartts import EarTTSInputEmbedding

import torch
from omegaconf import OmegaConf

# load tokens to be used for inference
prompt_audio_codes = torch.load("eartts_debug_tokens/code.pt").cuda()
next_subword_ids = torch.load("eartts_debug_tokens/next_subword_ids.pt").cuda()
full_subword_ids_prompt = torch.load("eartts_debug_tokens/input_text_tokens.pt").cuda()
subword_ids = full_subword_ids_prompt[0, :-1]  # T
shifted_prompt_audio_codes = torch.nn.functional.pad(prompt_audio_codes[0, :-1, :], (0, 0, 1, 0)).transpose(0, 1)  # T x 31 
next_subword_ids = torch.load("eartts_debug_tokens/next_subword_ids.pt").cuda()  # T

# load vllm engine
type_str = "float16"
torch_type = getattr(torch, type_str)
engine_args = AsyncEngineArgs(
    model="eartts_vllm_model",
    dtype=type_str,
    max_model_len=256,
    gpu_memory_utilization=0.6,
    enable_prompt_embeds=True,
    return_hidden_states=True,
    skip_tokenizer_init=True,  # Skip tokenizer since we're using embeddings directly
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=64, temperature=1.0)


# load embedding model that we use to create hidden states for vllm inference
conf = OmegaConf.load("eartts_vllm_model/eartts_input_embedding_config.yaml")
conf.model_dir = "eartts_vllm_model"
embedding = EarTTSInputEmbedding(conf)
embedding.load_state_dict(torch.load("eartts_vllm_model/eartts_input_embedding.ckpt"))
embedding.cuda()
embedding.to(torch_type)


# embeddings for context phase
prompt_embeds = embedding(
    context_text_tokens=subword_ids,
    audio_tokens=shifted_prompt_audio_codes,
).squeeze(0).detach().cpu().to(torch_type)
inputs = {"prompt_embeds": prompt_embeds}
acoustic_tokens = []
i = 0
async for output in engine.generate(inputs, sampling_params=sampling_params, request_id="1", is_streaming=True):
    hidden_states = output.outputs[0].hidden_states[-1]  # T x 31

    step_acoustic_tokens = hidden_states[-1].clone().to(torch.long)  # 31,
    acoustic_tokens.append(step_acoustic_tokens)

    # if previously prepared input was last, break
    if i == next_subword_ids.shape[1] - 1:
        break

    # prepare next input
    if i == 0:
        context_subword_id = full_subword_ids_prompt[:, -1]  # (1,)
    else:
        context_subword_id = next_subword_ids[:, i - 1]  # (1,)
    current_subword_id = next_subword_ids[:, i]  # (1,)
    next_input = embedding(
        context_text_tokens=context_subword_id.cuda(),
        audio_tokens=step_acoustic_tokens.unsqueeze(1).cuda(),
        text_tokens=current_subword_id.cuda(),
    )
    await engine.append_request(request_id="1", input_embeds=next_input.detach().cpu().to(torch_type))
    i += 1

acoustic_tokens_arr = torch.stack(acoustic_tokens)
torch.save(acoustic_tokens_arr.to(torch.long).detach().cpu(), "pred_tokens.pt")